# R1 Tools as UC Functions

In [0]:
create or replace function claims.genai.lookup_claims(claim_id string)
returns table (
    claim_id STRING,
  member_id STRING,
  hospital_id STRING,
  status STRING,
  claim_type STRING,
  admission_date DATE,
  discharge_date DATE,
  billed_amount DOUBLE,
  approved_amount DOUBLE,
  rejection_code STRING,
  is_pre_existing BOOLEAN,
  plan_code_at_admission STRING,
  plan_family STRING,
  room_rent_per_day BIGINT,
  room_rent_cap_per_day INT,
  room_rent_over_cap BOOLEAN,
  policy_start_date DATE,
  policy_tenure_months INT,
  days_since_policy_start INT,
  is_network_at_admission BOOLEAN
)

comment "Returns a claim's stored facts exactly as recorded in claims..gold.claims_enriched with no recomputation and no payout decision. Use this to look up what is already known about a specific claim."

return
    select 
    claim_id, member_id, hospital_id, status, claim_type,
    admission_date, discharge_date, billed_amount, approved_amount,
    rejection_code, is_pre_existing, plan_code_at_admission, plan_family,
    room_rent_per_day, room_rent_cap_per_day, room_rent_over_cap,
    policy_start_date, policy_tenure_months,
    datediff(admission_date, policy_start_date) AS days_since_policy_start,
    is_network_at_admission
    from claims.gold.claims_enriched
    where claims_enriched.claim_id = lookup_claims.claim_id;

In [0]:
create or replace function claims.genai.plan_at(member_id STRING, as_of_date DATE)
returns table (
  member_id STRING,
  policy_id STRING,
  plan_code STRING,
  plan_family STRING,
  room_rent_cap_per_day INT,
  copay_pct INT,
  ped_waiting_months INT,
  initial_waiting_days INT,
  valid_from DATE,
  valid_to DATE
)
comment 'Returns the policy version and plan terms valid for this member on the given date, compared by date against the policy history.'
return
  select
    scd.member_id,
    scd.policy_id,
    scd.plan_code,
    p.plan_family,
    p.room_rent_cap_per_day,
    p.copay_pct,
    p.ped_waiting_months,
    p.initial_waiting_days,
    scd.effective_from,
    scd.effective_till
  FROM claims.silver.dim_policy_scd2 scd
  JOIN claims.raw.plans p ON scd.plan_code = p.plan_code   
  WHERE scd.member_id = plan_at.member_id
    AND plan_at.as_of_date >= scd.effective_from
    AND (plan_at.as_of_date <= scd.effective_till OR scd.effective_till IS NULL);

In [0]:

CREATE OR REPLACE FUNCTION claims.genai.search_policy(query STRING)

RETURNS TABLE (
    chunk_id STRING,
    doc_name STRING,
    heading STRING,
    text STRING
)

COMMENT 'Search the policy binder using hybrid keyword and semantic search. Returns up to five most relevant policy chunks containing the chunk ID, document name, heading, and text.'

RETURN

SELECT
    CAST(chunk_id AS STRING) AS chunk_id,
    doc_name,
    heading,
    text

FROM vector_search(
    index => 'claims.genai.policy_chunk_index',
    query_text => query,
    query_type => 'HYBRID',
    num_results => 5
);


In [0]:

CREATE OR REPLACE FUNCTION claims.genai.compute_payable(
    plan_code STRING,
    is_pre_existing BOOLEAN,
    policy_tenure_months INT,
    days_since_policy_start INT,
    is_network_at_admission BOOLEAN,
    room_rent_per_day BIGINT,
    billed_amount DOUBLE
)
RETURNS STRING
LANGUAGE SQL
READS SQL DATA
COMMENT 'Deterministically adjudicates an insurance claim using plan waiting periods, room rent limits and copay rules. Returns a JSON string with the payable decision and amount. Does not invoke an AI model.'

RETURN

WITH plan_terms AS (

    SELECT
        p.plan_code,
        p.room_rent_cap_per_day,
        p.copay_pct,
        p.initial_waiting_days,
        p.ped_waiting_months

    FROM claims.raw.plans p

    WHERE p.plan_code =
        compute_payable.plan_code

    LIMIT 1
),

calculation AS (

    SELECT
        plan_code,
        room_rent_cap_per_day,
        copay_pct,
        initial_waiting_days,
        ped_waiting_months,

        -- Check initial waiting period
        days_since_policy_start >= initial_waiting_days
        AS initial_waiting_ok,

        -- Check pre-existing disease waiting period
        (
            NOT is_pre_existing
            OR policy_tenure_months >= ped_waiting_months
        ) AS ped_waiting_ok,

        -- Check room rent limit
        room_rent_per_day > room_rent_cap_per_day
        AS room_rent_over_cap,

        -- Calculate proportionate room rent deduction
        CASE
            WHEN room_rent_per_day > room_rent_cap_per_day
            THEN ROUND(
                (
                    (room_rent_per_day - room_rent_cap_per_day)
                    / CAST(room_rent_per_day AS DOUBLE)
                ) * 100,
                2
            )
            ELSE 0.00
        END AS room_rent_deduction_pct

    FROM plan_terms
),

final_calculation AS (

    SELECT
        *,

        (initial_waiting_ok AND ped_waiting_ok)
        AS waiting_period_ok,

        CASE
            WHEN NOT initial_waiting_ok
                THEN 'INITIAL_WAITING'

            WHEN NOT ped_waiting_ok
                THEN 'PED_WAITING'

            WHEN room_rent_deduction_pct > 0
                 OR copay_pct > 0
                THEN 'PAYABLE_WITH_DEDUCTION'

            ELSE 'PAYABLE_IN_FULL'
        END AS deduction_code

    FROM calculation
)

SELECT TO_JSON(
    NAMED_STRUCT(

        'plan_code', plan_code,

        'room_rent_cap_per_day',
            room_rent_cap_per_day,

        'room_rent_over_cap',
            room_rent_over_cap,

        'waiting_period_ok',
            waiting_period_ok,

        'payable',
            waiting_period_ok,

        'network_at_admission',
            is_network_at_admission,

        'room_rent_deduction_pct',
            room_rent_deduction_pct,

        'copay_pct',
            copay_pct,

        'deduction_code',
            deduction_code,

        'payable_amount',
            ROUND(
                CASE
                    WHEN waiting_period_ok = FALSE
                    THEN 0.00

                    ELSE billed_amount
                        * (1 - room_rent_deduction_pct / 100)
                        * (1 - copay_pct / 100.0)
                END,
                2
            )
    )
)

FROM final_calculation;